# Web Scraping Workshop

## Proactieve Zorgbemiddeling: hoe lang wacht Nederland op zorg?

Sinds 1 januari 2026 geldt **proactieve zorgbemiddeling** in Nederland. Als je te lang op een specialist wacht, moet je zorgverzekeraar jou nu *actief benaderen* en helpen sneller terecht te komen. Je hoeft niet meer zelf achter ze aan te bellen.

Maar hoe erg zijn die wachttijden eigenlijk? En welke ziekenhuizen zijn de ergste boosdoeners?

In deze workshop scrapen we wachttijden van **Zorgkaart Nederland**, ontdekken welke specialismen de norm overschrijden, en bouwen we een interactieve **Wachttijden Radar**.

### Wat gaan we doen?
1. Een webpagina ophalen met `requests`
2. HTML parsen met `BeautifulSoup`
3. Wachttijden per ziekenhuis uit een tabel halen
4. Meerdere specialismen scrapen
5. Vergelijken met de **treeknormen** (maximale wachttijden)
6. Een interactieve **Wachttijden Radar** bouwen

In [ ]:
# Eerst wat context over de wet en de treeknormen

## Achtergrond: wat zijn treeknormen?

De overheid heeft maximale wachttijden vastgesteld voor medisch-specialistische zorg — de **treeknormen**:

| Type | Maximale wachttijd |
|------|-------------------|
| Polikliniekbezoek | 4 weken (28 dagen) |
| Diagnostisch onderzoek | 4 weken (28 dagen) |
| Behandeling | 7 weken (49 dagen) |

Overschrijdt een ziekenhuis deze normen? Dan moet je verzekeraar sinds 2026 *proactief* contact met je opnemen en je elders plaatsen. Dat is de kern van **proactieve zorgbemiddeling**.

Laten we eens kijken hoe het er in de praktijk voor staat.

In [ ]:
%pip install requests beautifulsoup4 lxml pandas plotly

## Stap 1: Een wachttijdenpagina ophalen

Web scraping begint altijd hetzelfde: een HTTP-request naar een URL. We halen een pagina op van **Zorgkaart Nederland** die wachttijden per ziekenhuis toont voor cardiologie.

We gebruiken de library `requests` — precies wat je browser ook doet, maar dan zonder de mooie plaatjes.

In [ ]:
import requests

url = "https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2"

# We doen ons voor als een browser, anders blokkeert de server ons soms
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")  # 200 = succes!
print(f"Grootte: {len(response.text):,} tekens")
print(f"\nEerste 500 tekens:\n{response.text[:500]}")

## Stap 2: HTML parsen met BeautifulSoup

We hebben nu een lading HTML. **BeautifulSoup** parsed dit en laat ons zoeken met CSS selectors — net als in de browser DevTools.

> **💡 Tip:** Open https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2 in je browser, klik rechtermuisknop op een wachttijd → *Inspecteren*. Zo zie je de HTML-structuur.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "lxml")

# De wachttijden staan in een <table class="table table-certificates">
table = soup.select_one("table.table-certificates")
print(f"Tabel gevonden: {table is not None}")

# Hoeveel rijen?
rows = table.select("tr")
print(f"Aantal rijen: {len(rows)} (1 header + {len(rows)-1} ziekenhuizen)")

# Laten we de header bekijken
header = [th.get_text(strip=True) for th in rows[0].select("th")]
print(f"Kolommen: {header}")

# En de eerste 3 data-rijen
print("\n--- Eerste 3 rijen ---")
for row in rows[1:4]:
    cells = [td.get_text(strip=True) for td in row.select("td")]
    print(cells)

## Stap 3: Data uit de tabel halen

Elke rij in de tabel heeft drie kolommen:

```
tr
├── td  → Zorgaanbieder (naam + link naar profiel)
├── td  → Locatie (stad)
└── td  → Wachttijd in dagen
```

We halen elk stukje data op met CSS selectors.

In [ ]:
# Laten we één rij helemaal uitpluizen
row = rows[1]  # eerste data-rij
cells = row.select("td")

# Naam
naam = cells[0].get_text(strip=True)
print(f"Naam: {naam}")

# Link naar profiel
link_el = cells[0].select_one("a")
profiel_url = link_el["href"] if link_el else ""
print(f"Profiel: https://www.zorgkaartnederland.nl{profiel_url}")

# Locatie
locatie = cells[1].get_text(strip=True)
print(f"Locatie: {locatie}")

# Wachttijd (in dagen)
wachttijd_tekst = cells[2].get_text(strip=True)
print(f"Wachttijd: {wachttijd_tekst} dagen")

## Stap 4: Alle ziekenhuizen van één specialisme scrapen

We stoppen de logica hierboven in een functie die de complete tabel uitleest.

In [ ]:
def scrape_wachttijden(url):
    """Scrape alle wachttijden van één Zorgkaart Nederland-pagina."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")
    
    table = soup.select_one("table.table-certificates")
    if not table:
        return []
    
    resultaten = []
    for row in table.select("tr")[1:]:  # skip header
        cells = row.select("td")
        if len(cells) < 3:
            continue
        
        link_el = cells[0].select_one("a")
        wachttijd_tekst = cells[2].get_text(strip=True)
        
        resultaten.append({
            "zorgaanbieder": cells[0].get_text(strip=True),
            "locatie": cells[1].get_text(strip=True),
            "wachttijd_dagen": int(wachttijd_tekst) if wachttijd_tekst.isdigit() else None,
            "profiel_url": "https://www.zorgkaartnederland.nl" + link_el["href"] if link_el else "",
        })
    
    return resultaten

# Test!
resultaten = scrape_wachttijden("https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2")
print(f"Gevonden: {len(resultaten)} ziekenhuizen")
resultaten[:3]

## Stap 5: Meerdere specialismen scrapen

Eén specialisme is leuk, maar de echte kracht van scraping is schaal. We scrapen nu meerdere specialismen — zowel poliklinieken, diagnostiek als behandelingen — en bouwen een groot DataFrame.

De URL-structuur is simpel: `https://www.zorgkaartnederland.nl/wachttijden/{specialisme-slug}`

Met `time.sleep()` zijn we netjes voor de server.

In [ ]:
import time
import pandas as pd

# Selectie van specialismen per type wachttijd
specialismen = {
    # Poliklinieken (treeknorm: 28 dagen)
    "cardiologie-2": ("Cardiologie", "polikliniek"),
    "chirurgie-heelkunde": ("Chirurgie", "polikliniek"),
    "dermatologie": ("Dermatologie", "polikliniek"),
    "neurologie": ("Neurologie", "polikliniek"),
    "oogheelkunde": ("Oogheelkunde", "polikliniek"),
    "orthopedie-2": ("Orthopedie", "polikliniek"),
    # Diagnostiek (treeknorm: 28 dagen)
    "ct": ("CT-scan", "diagnostiek"),
    "echografie-radiologie": ("Echografie", "diagnostiek"),
    "mri-radiologie": ("MRI", "diagnostiek"),
    # Behandelingen (treeknorm: 49 dagen)
    "staaroperatie-oogheelkunde": ("Staaroperatie", "behandeling"),
    "totale-knievervanging-knieprothese-orthopedie": ("Knievervanging", "behandeling"),
    "totale-heupvervanging-heupprothese-orthopedie": ("Heupvervanging", "behandeling"),
}

alle_data = []
for slug, (naam, wtype) in specialismen.items():
    url = f"https://www.zorgkaartnederland.nl/wachttijden/{slug}"
    resultaten = scrape_wachttijden(url)
    for r in resultaten:
        r["specialisme"] = naam
        r["type"] = wtype
    alle_data.extend(resultaten)
    print(f"  {naam}: {len(resultaten)} ziekenhuizen")
    time.sleep(0.5)  # wees netjes

df = pd.DataFrame(alle_data)
print(f"\nTotaal: {len(df)} rijen, {df['specialisme'].nunique()} specialismen")
df.head(10)

## Stap 6: Treeknorm-analyse

Nu wordt het spannend. We vergelijken de werkelijke wachttijden met de treeknormen.

Waar moet je zorgverzekeraar eigenlijk aan de bel trekken?

In [ ]:
# Treeknormen in dagen
treeknormen = {
    "polikliniek": 28,   # 4 weken
    "diagnostiek": 28,   # 4 weken
    "behandeling": 49,   # 7 weken
}

# Voeg treeknorm toe per rij
df["treeknorm_dagen"] = df["type"].map(treeknormen)

# Overschrijdt dit ziekenhuis de norm?
df["overschrijding"] = df["wachttijd_dagen"] > df["treeknorm_dagen"]

# Statistieken
n_totaal = len(df.dropna(subset=["wachttijd_dagen"]))
n_overschrijding = df["overschrijding"].sum()
pct = n_overschrijding / n_totaal * 100

print(f"{n_overschrijding} van {n_totaal} ziekenhuizen ({pct:.0f}%) overschrijdt de treeknorm!")
print(f"\nPer specialisme:")
print(df.groupby("specialisme")["overschrijding"].mean().mul(100).round(0).sort_values(ascending=False).to_string())

In [ ]:
# Top 15 langste wachttijden
print("De 15 langste wachttijden in onze dataset:\n")
top15 = df.nlargest(15, "wachttijd_dagen")[["specialisme", "zorgaanbieder", "locatie", "wachttijd_dagen", "treeknorm_dagen"]].copy()
top15["overschrijding_dagen"] = top15["wachttijd_dagen"] - top15["treeknorm_dagen"]
print(top15.to_string(index=False))

## Stap 7: Visualiseren

### Percentage treeknorm-overschrijdingen per specialisme

In [ ]:
import plotly.express as px

# Percentage overschrijdingen per specialisme
pct_per_spec = (
    df.groupby(["specialisme", "type"])["overschrijding"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"overschrijding": "pct_overschrijding"})
    .sort_values("pct_overschrijding", ascending=True)
)

fig = px.bar(
    pct_per_spec,
    x="pct_overschrijding",
    y="specialisme",
    color="type",
    orientation="h",
    title="Percentage ziekenhuizen dat de treeknorm overschrijdt",
    labels={"pct_overschrijding": "% overschrijding", "specialisme": "", "type": "Type"},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
fig.add_vline(x=50, line_dash="dash", line_color="red", annotation_text="50%")
fig.update_layout(height=500)
fig.show()

### Gemiddelde wachttijd vs. treeknorm

In [ ]:
gem = (
    df.groupby(["specialisme", "type"])
    .agg(gem_wachttijd=("wachttijd_dagen", "mean"), treeknorm=("treeknorm_dagen", "first"))
    .reset_index()
    .sort_values("gem_wachttijd", ascending=True)
)

fig = px.bar(
    gem,
    x="gem_wachttijd",
    y="specialisme",
    orientation="h",
    title="Gemiddelde wachttijd per specialisme (in dagen)",
    labels={"gem_wachttijd": "Gemiddelde wachttijd (dagen)", "specialisme": ""},
    color="type",
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)

# Treeknorm-lijnen
fig.add_vline(x=28, line_dash="dash", line_color="#F44336", annotation_text="Treeknorm poli/diag (28d)")
fig.add_vline(x=49, line_dash="dash", line_color="#F44336", annotation_text="Treeknorm behandeling (49d)")
fig.update_layout(height=500)
fig.show()

### Spreiding per specialisme

Het gemiddelde vertelt niet het hele verhaal. Hoe groot is de spreiding?

In [ ]:
fig = px.box(
    df,
    x="wachttijd_dagen",
    y="specialisme",
    color="type",
    orientation="h",
    title="Spreiding wachttijden per specialisme",
    labels={"wachttijd_dagen": "Wachttijd (dagen)", "specialisme": ""},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
fig.add_vline(x=28, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.add_vline(x=49, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.update_layout(height=600)
fig.show()

## Stap 8: Interactieve Wachttijden Radar

Nu het eindproduct! We bouwen een interactieve **Wachttijden Radar**: kies een specialisme en zie direct welke ziekenhuizen de kortste wachttijden hebben, en welke de treeknorm overschrijden.

Het resultaat slaan we op als een standalone HTML-bestand dat je kunt delen.

In [ ]:
# TODO: Interactieve Wachttijden Radar
# (plotly code hier - zie workshop voor volledige versie)
print('Radar wordt hier gebouwd!')

In [ ]:
# Sla de radar op als standalone HTML
fig.write_html("wachttijden_radar.html", include_plotlyjs=True)
print("Opgeslagen: wachttijden_radar.html")
print("Open dit bestand in je browser voor de interactieve Wachttijden Radar!")

# En de ruwe data als CSV
df.to_csv("wachttijden_zorg.csv", index=False)
print(f"\nData opgeslagen: wachttijden_zorg.csv ({len(df)} rijen)")

## Recap

| Wat | Hoe |
|-----|-----|
| Webpagina ophalen | `requests.get(url)` |
| HTML parsen | `BeautifulSoup(html, "lxml")` |
| Tabel zoeken | `soup.select_one("table.table-certificates")` |
| Rijen doorlopen | `table.select("tr")[1:]` |
| Cellen uitlezen | `row.select("td")` + `.get_text(strip=True)` |
| Link uitlezen | `cell.select_one("a")["href"]` |
| Meerdere pagina’s | Loop + `time.sleep()` |
| Data structureren | `pd.DataFrame(lijst_van_dicts)` |
| Interactieve viz | `plotly` + `fig.write_html()` |

### Zelf verder experimenteren?

- Meer specialismen toevoegen: bekijk alle opties op [/wachttijden/poliklinieken](https://www.zorgkaartnederland.nl/wachttijden/poliklinieken)
- De overzichtspagina `/wachttijden` scrapen om *alle* specialisme-links automatisch te vinden
- Per ziekenhuis de detailpagina scrapen voor wachttijden over alle specialismen heen
- Wachttijden over tijd vergelijken (de NZa publiceert elke 2 weken nieuwe data)
- Combineer met de huisartsscores uit de vorige workshop: scoren ziekenhuizen met korte wachttijden ook beter?